<a href="https://colab.research.google.com/github/AymaneAmri-MD/drug-design-morphine-paclitaxel/blob/main/notebooks/02_docking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# 1. Installation des dépendances et téléchargement du binaire officiel AutoDock Vina v1.2.5
!apt-get update -qq
!apt-get install -y -qq openbabel
!pip install -q biopython pandas

# Descarga directa del binario ejecutable sin compilación C++
!wget -q https://github.com/CCSB-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O /usr/local/bin/vina
!chmod +x /usr/local/bin/vina

import os
import sys

# Positionnement dans le répertoire notebooks
if os.path.basename(os.getcwd()) != 'notebooks':
    if os.path.exists('drug-design-morphine-paclitaxel'):
        %cd drug-design-morphine-paclitaxel/notebooks
    else:
        !git clone -q https://github.com/AymaneAmri-MD/drug-design-morphine-paclitaxel.git
        %cd drug-design-morphine-paclitaxel/notebooks

# Création des dossiers de travail
os.makedirs('../data/pdb', exist_ok=True)
os.makedirs('../results/docking', exist_ok=True)

print("Environnement configuré et binaire AutoDock Vina v1.2.5 prêt !")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-iostreams1.83.0:amd64.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.83.0_1.83.0-2.1ubuntu3.2_amd64.deb ...
Unpacking libboost-iostreams1.83.0:amd64 (1.83.0-2.1ubuntu3.2) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4build1_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4build1) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.3.1-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.3.1-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-9ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-9ubuntu5) ...
S

In [4]:
# 2. Téléchargement des structures PDB et suppression des molécules d'eau
import os
import urllib.request
from Bio import PDB

os.makedirs('../data/pdb', exist_ok=True)

pdb_ids = {'morphine_target': '5C1M', 'paclitaxel_target': '1JFF'}

def download_pdb(pdb_id, output_path):
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    urllib.request.urlretrieve(url, output_path)
    print(f"PDB {pdb_id} téléchargé -> {output_path}")

# Téléchargement des fichiers bruts
for key, pdb_id in pdb_ids.items():
    raw_pdb = f"../data/pdb/{pdb_id}.pdb"
    download_pdb(pdb_id, raw_pdb)

# Filtre pour exclure les molécules d'eau (HOH)
class NonWaterSelect(PDB.Select):
    def accept_residue(self, residue):
        return residue.get_resname() != 'HOH'

parser = PDB.PDBParser(QUIET=True)
io = PDB.PDBIO()

# Nettoyage des fichiers PDB
for key, pdb_id in pdb_ids.items():
    struct = parser.get_structure(pdb_id, f"../data/pdb/{pdb_id}.pdb")
    clean_path = f"../data/pdb/{pdb_id}_clean.pdb"
    io.set_structure(struct)
    io.save(clean_path, NonWaterSelect())
    print(f"Structure nettoyée sans eau -> {clean_path}")

PDB 5C1M téléchargé -> ../data/pdb/5C1M.pdb
PDB 1JFF téléchargé -> ../data/pdb/1JFF.pdb
Structure nettoyée sans eau -> ../data/pdb/5C1M_clean.pdb
Structure nettoyée sans eau -> ../data/pdb/1JFF_clean.pdb


In [17]:
# 3. Préparation des récepteurs et régénération/conversion des ligands en PDBQT
import os
import subprocess

os.makedirs('../results/docking', exist_ok=True)

# 1. Conversion des récepteurs : PDB -> PDBQT
for pdb_id in ['5C1M', '1JFF']:
    input_pdb = f"../data/pdb/{pdb_id}_clean.pdb"
    output_pdbqt = f"../results/docking/{pdb_id}_receptor.pdbqt"
    subprocess.run(["obabel", "-ipdb", input_pdb, "-opdbqt", "-O", output_pdbqt, "-xr", "-h", "--partialcharge", "gasteiger"], check=True)
    print(f"Récepteur PDBQT prêt -> {output_pdbqt}")

# 2. SMILES de secours si les fichiers .mol sont manquants ou vides
smiles_dict = {
    'morphine': 'CN1CCC23c4c5ccc(c4O2)C(C1C3)O',
    'paclitaxel': 'CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)C(C(C5=CC=CC=C5)NC(=O)C6=CC=CC=C6)O)O)OC(=O)C7=CC=CC=C7)(CO4)OC(=O)C)O)C)OC(=O)C'
}

ligands = {
    'morphine': '../results/morphine_best.mol',
    'paclitaxel': '../results/paclitaxel_best.mol'
}

for name, mol_path in ligands.items():
    out_pdbqt = f"../results/docking/{name}_ligand.pdbqt"

    # Si le fichier .mol n'existe pas ou est vide (< 100 octets), on génère le 3D directement depuis SMILES
    if not os.path.exists(mol_path) or os.path.getsize(mol_path) < 100:
        print(f"Régénération 3D directe depuis SMILES pour {name}...")
        smi = smiles_dict[name]
        cmd = ["obabel", f"-:{smi}", "-opdbqt", "-O", out_pdbqt, "--gen3d", "-h", "--partialcharge", "gasteiger"]
    else:
        cmd = ["obabel", "-imol", mol_path, "-opdbqt", "-O", out_pdbqt, "-h", "--partialcharge", "gasteiger"]

    subprocess.run(cmd, check=True)
    print(f"Ligand PDBQT prêt -> {out_pdbqt} (Taille : {os.path.getsize(out_pdbqt)} octets)")

Récepteur PDBQT prêt -> ../results/docking/5C1M_receptor.pdbqt
Récepteur PDBQT prêt -> ../results/docking/1JFF_receptor.pdbqt
Régénération 3D directe depuis SMILES pour morphine...
Ligand PDBQT prêt -> ../results/docking/morphine_ligand.pdbqt (Taille : 0 octets)
Régénération 3D directe depuis SMILES pour paclitaxel...
Ligand PDBQT prêt -> ../results/docking/paclitaxel_ligand.pdbqt (Taille : 7246 octets)


In [18]:
# 4. Exécution du docking AutoDock Vina v1.2.5
import os
import subprocess

os.makedirs('../results/docking', exist_ok=True)

docking_configs = {
    'morphine': {
        'receptor': '../results/docking/5C1M_receptor.pdbqt',
        'ligand': '../results/docking/morphine_ligand.pdbqt',
        'center': [-12.5, 18.2, -8.4],
        'size': [20.0, 20.0, 20.0],
        'out': '../results/docking/morphine_docked.pdbqt',
        'log': '../results/docking/morphine.log'
    },
    'paclitaxel': {
        'receptor': '../results/docking/1JFF_receptor.pdbqt',
        'ligand': '../results/docking/paclitaxel_ligand.pdbqt',
        'center': [20.1, -15.3, 12.8],
        'size': [25.0, 25.0, 25.0],
        'out': '../results/docking/paclitaxel_docked.pdbqt',
        'log': '../results/docking/paclitaxel.log'
    }
}

for name, config in docking_configs.items():
    print(f"--- Lancement du docking pour {name} ---")

    cmd = [
        "vina",
        "--receptor", config['receptor'],
        "--ligand", config['ligand'],
        "--center_x", str(config['center'][0]),
        "--center_y", str(config['center'][1]),
        "--center_z", str(config['center'][2]),
        "--size_x", str(config['size'][0]),
        "--size_y", str(config['size'][1]),
        "--size_z", str(config['size'][2]),
        "--exhaustiveness", "8",
        "--out", config['out']
    ]

    # Exécution et redirection de la sortie vers le fichier .log
    with open(config['log'], 'w') as log_file:
        result = subprocess.run(cmd, stdout=log_file, stderr=subprocess.PIPE, text=True)

    if result.returncode == 0:
        print(f"Docking {name} terminé avec succès ! Résultat : {config['out']}")
    else:
        print(f"Erreur lors du docking de {name} : {result.stderr}")

--- Lancement du docking pour morphine ---
Erreur lors du docking de morphine : 

PDBQT parsing error: No atoms in this ligand.

--- Lancement du docking pour paclitaxel ---


KeyboardInterrupt: 

In [19]:
# 5. Extraction des énergies de liaison (kcal/mol) et analyse des résultats
import re
import os
import pandas as pd

results = []

docking_logs = {
    'morphine': '../results/docking/morphine.log',
    'paclitaxel': '../results/docking/paclitaxel.log'
}

for name, log_file in docking_logs.items():
    if os.path.exists(log_file):
        with open(log_file, 'r') as f:
            log_content = f.read()
            # Extraction du meilleur score (pose 1)
            match = re.search(r'^\s*1\s+(-?\d+\.\d+)', log_content, re.MULTILINE)
            score = float(match.group(1)) if match else None
            results.append({'Molecule': name, 'Affinity_kcal_mol': score})
    else:
        results.append({'Molecule': name, 'Affinity_kcal_mol': "Log manquant"})

df_results = pd.DataFrame(results)
summary_path = '../results/docking_summary.csv'
df_results.to_csv(summary_path, index=False)

print("=== RÉSULTATS DU DOCKING MOLÉCULAIRE ===")
print(df_results)

=== RÉSULTATS DU DOCKING MOLÉCULAIRE ===
     Molecule Affinity_kcal_mol
0    morphine              None
1  paclitaxel              None
